Proses Import Excel

In [213]:
!pip install xlsxwriter
import pandas as pd

# jika file berada di root direktori Colab, cukup tulis 'nama_file_anda.xlsx'
# Jika file Anda ada di Google Drive, Anda perlu melakukan mount Google Drive terlebih dahulu.
file_path = 'upload_data_products.xlsx'

try:
    # Membaca file Excel ke dalam DataFrame pandas
    df = pd.read_excel(file_path)

    print("File Excel berhasil dibaca!")
    print("5 baris pertama dari DataFrame:")
    print(df.head())

except FileNotFoundError:
    print(f"Error: File '{file_path}' tidak ditemukan. Pastikan nama file dan path sudah benar dan file sudah diunggah.")
except Exception as e:
    print(f"Terjadi kesalahan saat membaca atau mengolah file Excel: {e}")


File Excel berhasil dibaca!
5 baris pertama dari DataFrame:
                                                name   price           brand  \
0  refill studio tropik dreamsetter airbrush sett...   99900   studio tropik   
1                     in perfect pair cushion powder  166500        gladglow   
2            hr oil control pearlfect cover skintint  122850  mad for makeup   
3                        acne th longwear foundation   89700      sea makeup   
4                acne armor cover  correct skin tint   39920       dazzle me   

                                              shades  \
0                      Translucent neutral cool warm   
1  Affogato Buttercream Praline Custard Ginger Ci...   
2  Shade N1 Shade N2 Shade N1.5 Shade W0.5 Shade ...   
3  Chocolate Dip Caramel Cookie Dough Peanut Butt...   
4  Light Ivory Light Bone Yellow Beige Medium Pin...   

                                         ingredients  \
0  synthetic fluorphlogopite mica silica zinc myr...   
1  cushion

Preprocessing Nama Produk

In [214]:
nama_produk = df['name']
# print("List Product:")
# print(nama_produk)
# print("=============================================")

nama_produk_split = []
for product_name in nama_produk:
    nama_produk_split.append(product_name.split())
# print("List Splited Nama Product:")
# print(nama_produk_split)
# print("=============================================")

nama_produk_split_join = [item for sublist in nama_produk_split for item in sublist]
# print("List Splited Nama Product Join All:")
# print(nama_produk_split_join)
# print("Total: ", len(nama_produk_split_join))
# print("=============================================")

nama_produk_split_join_unique = list(dict.fromkeys(nama_produk_split_join))

# Sort by alphabet
nama_produk_split_join_unique.sort()

# print("List Splited Nama Product Join All Unique:")
# print(nama_produk_split_join_unique)
# print("Total: ", len(nama_produk_split_join_unique))
# print("=============================================")

# Make tabel with thead is nama_produk_split_join_unique and tbody is count of nama_produk_split containing each thead
import pandas as pd

# Create a list of all columns, starting with 'Product Number' and 'Product Name'. Then total column in last
all_columns = ['Product Number', 'Product Name'] + nama_produk_split_join_unique + ['Total']
df_nama_produk = pd.DataFrame(columns=all_columns)
total_each_product = []

for i, product_split_list in enumerate(nama_produk_split):
    # Get the original product name for the current row
    original_product_name = nama_produk.iloc[i]
    numbering_product = "P" + str(i+1)

    # Calculate counts for each unique word
    counts = [product_split_list.count(item) for item in nama_produk_split_join_unique]

    # Generate Total Column
    total = sum(counts)
    counts.append(total)

    # Appen product name and its total
    total_each_product.append([original_product_name, total])

    # Combine product number, original product name with counts
    row_data = [numbering_product, original_product_name] + counts

    # Add the row to the DataFrame
    df_nama_produk.loc[len(df_nama_produk)] = row_data

# Append row total for each column and custom string for column 'Product Number', 'Product Name'
list_idf = len(nama_produk) / df_nama_produk.sum(numeric_only=True, axis=0)
df_nama_produk.loc[len(df_nama_produk)] = list_idf
df_nama_produk.at[len(df_nama_produk)-1, 'Product Number'] = 'IDF'
df_nama_produk.at[len(df_nama_produk)-1, 'Product Name'] = ''

# Term Frequency
all_columns_term_frequency = ['Product Number', 'Product Name'] + nama_produk_split_join_unique
df_nama_produk_term_frequency = pd.DataFrame(columns=all_columns_term_frequency)

for i, product_split_list in enumerate(nama_produk_split):
    # Get the original product name for the current row
    original_product_name = nama_produk.iloc[i]
    numbering_product = "P" + str(i+1)

    product_total_words = 0
    for prod_name, total_words in total_each_product:
        if prod_name == original_product_name:
            product_total_words = total_words
            break

    # Calculate counts for each unique word, then divided by total_each_product for Term Frequency
    # Ensure to handle division by zero if a product has no words (though unlikely here)
    counts = [(product_split_list.count(item) / product_total_words) if product_total_words > 0 else 0 for item in nama_produk_split_join_unique]

    # Combine product number, original product name with counts
    row_data = [numbering_product, original_product_name] + counts

    # Add the row to the DataFrame
    df_nama_produk_term_frequency.loc[len(df_nama_produk_term_frequency)] = row_data

# Term Frequency - Inverse Document Frequency
df_nama_produk_inverse_doc_frequency = df_nama_produk_term_frequency.copy() # Use .copy() to avoid SettingWithCopyWarning

# Loop through data frame then divide its cell with list_idf
for column in nama_produk_split_join_unique:
    if column in list_idf.index:
        df_nama_produk_inverse_doc_frequency[column] = df_nama_produk_inverse_doc_frequency[column] * list_idf[column]

# print("List product extraction")
# print(df_nama_produk)
# print("=============================================")
# print("=============================================")
# print("List product term frequency")
# print(df_nama_produk_term_frequency)
# print("=============================================")
# print("=============================================")
# print("List product inverse document frequency")
# print(df_nama_produk_inverse_doc_frequency)
# print("=============================================")
# print("=============================================")
# print("List IDF")
# print(list_idf)

# Export Excel to a single sheet
# Convert list_idf to a DataFrame for easier writing
list_idf_df = list_idf.rename('IDF Value').to_frame().T
list_idf_df.insert(0, 'Product Number', 'IDF')
list_idf_df.insert(1, 'Product Name', '')

with pd.ExcelWriter('product_preprocessing.xlsx') as writer:
    start_row = 0

    # Write Product Extraction Data
    pd.DataFrame([['Product Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_produk) + 2

    # Write Term Frequency Data
    pd.DataFrame([['Term Frequency Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk_term_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_produk_term_frequency) + 2

    # Write TF-IDF Data
    pd.DataFrame([['TF-IDF Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk_inverse_doc_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_produk_inverse_doc_frequency) + 2

print("results are exported to product_preprocessing.xlsx")


results are exported to product_preprocessing.xlsx


Preporcessing Ingredients

In [215]:
nama_ingredient = df['ingredients']

nama_ingredient_split = []
for ingredient_name in nama_ingredient:
    nama_ingredient_split.append(ingredient_name.split())

nama_ingredient_split_join = [item for sublist in nama_ingredient_split for item in sublist]
nama_ingredient_split_join_unique = list(dict.fromkeys(nama_ingredient_split_join))

# Sort by alphabet
nama_ingredient_split_join_unique.sort()

# Make tabel with thead is nama_ingredient_split_join_unique and tbody is count of nama_ingredient_split containing each thead
import pandas as pd

# Create a list of all columns, starting with 'Ingredient Number' and 'Ingredient Name'. Then total column in last
all_columns = ['Ingredient Number', 'Ingredient Name'] + nama_ingredient_split_join_unique + ['Total']
df_nama_ingredient = pd.DataFrame(columns=all_columns)
total_each_ingredient = []

for i, ingredient_split_list in enumerate(nama_ingredient_split):
    # Get the original ingredient name for the current row
    original_ingredient_name = nama_ingredient.iloc[i]
    numbering_ingredient = "P" + str(i+1)

    # Calculate counts for each unique word
    counts = [ingredient_split_list.count(item) for item in nama_ingredient_split_join_unique]

    # Generate Total Column
    total = sum(counts)
    counts.append(total)

    # Appen ingredient name and its total
    total_each_ingredient.append([original_ingredient_name, total])

    # Combine ingredient number, original ingredient name with counts
    row_data = [numbering_ingredient, original_ingredient_name] + counts

    # Add the row to the DataFrame
    df_nama_ingredient.loc[len(df_nama_ingredient)] = row_data

# Append row total for each column and custom string for column 'Ingredient Number', 'Ingredient Name'
list_idf = len(nama_ingredient) / df_nama_ingredient.sum(numeric_only=True, axis=0)
df_nama_ingredient.loc[len(df_nama_ingredient)] = list_idf
df_nama_ingredient.at[len(df_nama_ingredient)-1, 'Ingredient Number'] = 'IDF'
df_nama_ingredient.at[len(df_nama_ingredient)-1, 'Ingredient Name'] = ''

# Term Frequency
all_columns_term_frequency = ['Ingredient Number', 'Ingredient Name'] + nama_ingredient_split_join_unique
df_nama_ingredient_term_frequency = pd.DataFrame(columns=all_columns_term_frequency)

for i, ingredient_split_list in enumerate(nama_ingredient_split):
    # Get the original ingredient name for the current row
    original_ingredient_name = nama_ingredient.iloc[i]
    numbering_ingredient = "P" + str(i+1)

    ingredient_total_words = 0
    for ingredient_name, total_words in total_each_ingredient:
        if ingredient_name == original_ingredient_name:
            ingredient_total_words = total_words
            break

    # Calculate counts for each unique word, then divided by total_each_ingredient for Term Frequency
    # Ensure to handle division by zero if a ingredient has no words (though unlikely here)
    counts = [(ingredient_split_list.count(item) / ingredient_total_words) if ingredient_total_words > 0 else 0 for item in nama_ingredient_split_join_unique]

    # Combine ingredient number, original ingredient name with counts
    row_data = [numbering_ingredient, original_ingredient_name] + counts

    # Add the row to the DataFrame
    df_nama_ingredient_term_frequency.loc[len(df_nama_ingredient_term_frequency)] = row_data

# Term Frequency - Inverse Document Frequency
df_nama_ingredient_inverse_doc_frequency = df_nama_ingredient_term_frequency.copy() # Use .copy() to avoid SettingWithCopyWarning

# Loop through data frame then divide its cell with list_idf
for column in nama_ingredient_split_join_unique:
    if column in list_idf.index:
        df_nama_ingredient_inverse_doc_frequency[column] = df_nama_ingredient_inverse_doc_frequency[column] * list_idf[column]

# Export Excel to a single sheet
# Convert list_idf to a DataFrame for easier writing
list_idf_df = list_idf.rename('IDF Value').to_frame().T
list_idf_df.insert(0, 'Ingredient Number', 'IDF')
list_idf_df.insert(1, 'Ingredient Name', '')

with pd.ExcelWriter('ingredient_preprocessing.xlsx') as writer:
    start_row = 0

    # Write Ingredient Extraction Data
    pd.DataFrame([['Ingredient Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient) + 2

    # Write Term Frequency Data
    pd.DataFrame([['Term Frequency Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient_term_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient_term_frequency) + 2

    # Write TF-IDF Data
    pd.DataFrame([['TF-IDF Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient_inverse_doc_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient_inverse_doc_frequency) + 2

print("results are exported to ingredient_preprocessing.xlsx")


results are exported to ingredient_preprocessing.xlsx


Brand Preprocessing

In [216]:
import math

nama_brand = df['brand']

nama_brand_split = []
for brand_name in nama_brand:
    nama_brand_split.append(brand_name.split())

nama_brand_split_join = [item for sublist in nama_brand_split for item in sublist]
nama_brand_split_join_unique = list(dict.fromkeys(nama_brand_split_join))

# Sort by alphabet
nama_brand_split_join_unique.sort()

# Make tabel with thead is nama_brand_split_join_unique and tbody is count of nama_brand_split containing each thead
import pandas as pd

# Create a list of all columns, starting with 'Brand Number' and 'Brand Name'. Then total column in last
all_columns = ['Brand Number', 'Brand Name'] + nama_brand_split_join_unique + ['Total']
df_nama_brand = pd.DataFrame(columns=all_columns)
total_each_brand = []

for i, brand_split_list in enumerate(nama_brand_split):
    # Get the original brand name for the current row
    original_brand_name = nama_brand.iloc[i]
    numbering_brand = "P" + str(i+1)

    # Calculate counts for each unique word
    counts = [brand_split_list.count(item) for item in nama_brand_split_join_unique]

    # Generate Total Column
    total = sum(counts)
    counts.append(total)

    # Appen brand name and its total
    total_each_brand.append([original_brand_name, total])

    # Combine brand number, original brand name with counts
    row_data = [numbering_brand, original_brand_name] + counts

    # Add the row to the DataFrame
    df_nama_brand.loc[len(df_nama_brand)] = row_data

# Append row total for each column and custom string for column 'Brand Number', 'Brand Name'. Then find each log10
list_idf = len(nama_brand) / df_nama_brand.sum(numeric_only=True, axis=0)
for column in list_idf.index:
    list_idf[column] = math.log10(list_idf[column])
df_nama_brand.loc[len(df_nama_brand)] = list_idf
df_nama_brand.at[len(df_nama_brand)-1, 'Brand Number'] = 'IDF'
df_nama_brand.at[len(df_nama_brand)-1, 'Brand Name'] = ''

# Term Frequency
all_columns_term_frequency = ['Brand Number', 'Brand Name'] + nama_brand_split_join_unique
df_nama_brand_term_frequency = pd.DataFrame(columns=all_columns_term_frequency)

for i, brand_split_list in enumerate(nama_brand_split):
    # Get the original brand name for the current row
    original_brand_name = nama_brand.iloc[i]
    numbering_brand = "P" + str(i+1)

    brand_total_words = 0
    for brand_name, total_words in total_each_brand:
        if brand_name == original_brand_name:
            brand_total_words = total_words
            break

    # Calculate counts for each unique word, then divided by total_each_brand for Term Frequency
    # Ensure to handle division by zero if a brand has no words (though unlikely here)
    counts = [(brand_split_list.count(item) / brand_total_words) if brand_total_words > 0 else 0 for item in nama_brand_split_join_unique]

    # Combine brand number, original brand name with counts
    row_data = [numbering_brand, original_brand_name] + counts

    # Add the row to the DataFrame
    df_nama_brand_term_frequency.loc[len(df_nama_brand_term_frequency)] = row_data

# Term Frequency - Inverse Document Frequency
df_nama_brand_inverse_doc_frequency = df_nama_brand_term_frequency.copy() # Use .copy() to avoid SettingWithCopyWarning

# Loop through data frame then divide its cell with list_idf
for column in nama_brand_split_join_unique:
    if column in list_idf.index:
        df_nama_brand_inverse_doc_frequency[column] = df_nama_brand_inverse_doc_frequency[column] * list_idf[column]

# Export Excel to a single sheet
# Convert list_idf to a DataFrame for easier writing
list_idf_df = list_idf.rename('IDF Value').to_frame().T
list_idf_df.insert(0, 'Brand Number', 'IDF')
list_idf_df.insert(1, 'Brand Name', '')

with pd.ExcelWriter('brand_preprocessing.xlsx') as writer:
    start_row = 0

    # Write Brand Extraction Data
    pd.DataFrame([['Brand Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_brand) + 2

    # Write Term Frequency Data
    pd.DataFrame([['Term Frequency Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand_term_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_brand_term_frequency) + 2

    # Write TF-IDF Data
    pd.DataFrame([['TF-IDF Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand_inverse_doc_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_brand_inverse_doc_frequency) + 2

print("results are exported to brand_preprocessing.xlsx")


results are exported to brand_preprocessing.xlsx


Shades Preprocessing

In [217]:
import math

nama_shade = df['shades']

nama_shade_split = []
for shade_name in nama_shade:
    nama_shade_split.append(shade_name.split())

nama_shade_split_join = [item for sublist in nama_shade_split for item in sublist]
nama_shade_split_join_unique = list(dict.fromkeys(nama_shade_split_join))

# Sort by alphabet
nama_shade_split_join_unique.sort()

# Make tabel with thead is nama_shade_split_join_unique and tbody is count of nama_shade_split containing each thead
import pandas as pd

# Create a list of all columns, starting with 'Shade Number' and 'Shade Name'. Then total column in last
all_columns = ['Shade Number', 'Shade Name'] + nama_shade_split_join_unique + ['Total']
df_nama_shade = pd.DataFrame(columns=all_columns)
total_each_shade = []

for i, shade_split_list in enumerate(nama_shade_split):
    # Get the original shade name for the current row
    original_shade_name = nama_shade.iloc[i]
    numbering_shade = "P" + str(i+1)

    # Calculate counts for each unique word
    counts = [shade_split_list.count(item) for item in nama_shade_split_join_unique]

    # Generate Total Column
    total = sum(counts)
    counts.append(total)

    # Appen shade name and its total
    total_each_shade.append([original_shade_name, total])

    # Combine shade number, original shade name with counts
    row_data = [numbering_shade, original_shade_name] + counts

    # Add the row to the DataFrame
    df_nama_shade.loc[len(df_nama_shade)] = row_data

# Append row total for each column and custom string for column 'Shade Number', 'Shade Name'. Then find each log10
list_idf = len(nama_shade) / df_nama_shade.sum(numeric_only=True, axis=0)
for column in list_idf.index:
    list_idf[column] = math.log10(list_idf[column])
df_nama_shade.loc[len(df_nama_shade)] = list_idf
df_nama_shade.at[len(df_nama_shade)-1, 'Shade Number'] = 'IDF'
df_nama_shade.at[len(df_nama_shade)-1, 'Shade Name'] = ''

# Term Frequency
all_columns_term_frequency = ['Shade Number', 'Shade Name'] + nama_shade_split_join_unique
df_nama_shade_term_frequency = pd.DataFrame(columns=all_columns_term_frequency)

for i, shade_split_list in enumerate(nama_shade_split):
    # Get the original shade name for the current row
    original_shade_name = nama_shade.iloc[i]
    numbering_shade = "P" + str(i+1)

    shade_total_words = 0
    for shade_name, total_words in total_each_shade:
        if shade_name == original_shade_name:
            shade_total_words = total_words
            break

    # Calculate counts for each unique word, then divided by total_each_shade for Term Frequency
    # Ensure to handle division by zero if a shade has no words (though unlikely here)
    counts = [(shade_split_list.count(item) / shade_total_words) if shade_total_words > 0 else 0 for item in nama_shade_split_join_unique]

    # Combine shade number, original shade name with counts
    row_data = [numbering_shade, original_shade_name] + counts

    # Add the row to the DataFrame
    df_nama_shade_term_frequency.loc[len(df_nama_shade_term_frequency)] = row_data

# Term Frequency - Inverse Document Frequency
df_nama_shade_inverse_doc_frequency = df_nama_shade_term_frequency.copy() # Use .copy() to avoid SettingWithCopyWarning

# Loop through data frame then divide its cell with list_idf
for column in nama_shade_split_join_unique:
    if column in list_idf.index:
        df_nama_shade_inverse_doc_frequency[column] = df_nama_shade_inverse_doc_frequency[column] * list_idf[column]

# Export Excel to a single sheet
# Convert list_idf to a DataFrame for easier writing
list_idf_df = list_idf.rename('IDF Value').to_frame().T
list_idf_df.insert(0, 'Shade Number', 'IDF')
list_idf_df.insert(1, 'Shade Name', '')

with pd.ExcelWriter('shade_preprocessing.xlsx') as writer:
    start_row = 0

    # Write Shade Extraction Data
    pd.DataFrame([['Shade Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_shade) + 2

    # Write Term Frequency Data
    pd.DataFrame([['Term Frequency Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade_term_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_shade_term_frequency) + 2

    # Write TF-IDF Data
    pd.DataFrame([['TF-IDF Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade_inverse_doc_frequency.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_shade_inverse_doc_frequency) + 2

print("results are exported to shade_preprocessing.xlsx")


results are exported to shade_preprocessing.xlsx


Category Preprocessing

In [218]:
nama_category = df['Category']

nama_category_split = []
for category_name in nama_category:
    nama_category_split.append(category_name.split())

nama_category_split_join = [item for sublist in nama_category_split for item in sublist]
nama_category_split_join_unique = list(dict.fromkeys(nama_category_split_join))

# Sort by alphabet
nama_category_split_join_unique.sort()

# Make tabel with thead is nama_category_split_join_unique and tbody is count of nama_category_split containing each thead
import pandas as pd

# Create a list of all columns, starting with 'Category Number' and 'Category Name'. Then total column in last
all_columns = ['Category Number', 'Category Name'] + nama_category_split_join_unique + ['Total']
df_nama_category = pd.DataFrame(columns=all_columns)
total_each_category = []

for i, category_split_list in enumerate(nama_category_split):
    # Get the original category name for the current row
    original_category_name = nama_category.iloc[i]
    numbering_category = "P" + str(i+1)

    # Calculate counts for each unique word
    counts = [category_split_list.count(item) for item in nama_category_split_join_unique]

    # Generate Total Column
    total = sum(counts)
    counts.append(total)

    # Appen category name and its total
    total_each_category.append([original_category_name, total])

    # Combine category number, original category name with counts
    row_data = [numbering_category, original_category_name] + counts

    # Add the row to the DataFrame
    df_nama_category.loc[len(df_nama_category)] = row_data

# Export Excel to a single sheet
with pd.ExcelWriter('category_preprocessing.xlsx') as writer:
    start_row = 0

    # Write Category Extraction Data
    pd.DataFrame([['Category Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_category.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_nama_category) + 2

print("results are exported to category_preprocessing.xlsx")


results are exported to category_preprocessing.xlsx


Price Preprocessing

In [219]:
import math

# Get name and price as dataframe, ensuring it's a copy to avoid SettingWithCopyWarning
df_name_price = df[['name', 'price']].copy()

# Calculate min and max price for normalization
min_price = df_name_price['price'].min()
max_price = df_name_price['price'].max()

# Apply the new normalization formula: (price - min_price) / (max_price - min_price)
# Handle division by zero if all prices are the same
if (max_price - min_price) != 0:
    df_name_price['normalized_price'] = (df_name_price['price'] - min_price) / (max_price - min_price)
else:
    df_name_price['normalized_price'] = 0.0 # Assign 0 if all prices are identical

# Display the DataFrame with the new column
display(df_name_price)

# Export Excel to a single sheet
with pd.ExcelWriter('price_preprocessing.xlsx') as writer:
    start_row = 0

    # Write price Extraction Data
    pd.DataFrame([['Price Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_name_price.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_name_price) + 2

print("results are exported to price_preprocessing.xlsx")

,name,price,normalized_price
0,refill studio tropik dreamsetter airbrush sett...,99900,0.473851
1,in perfect pair cushion powder,166500,1.000000
2,hr oil control pearlfect cover skintint,122850,0.655159
3,acne th longwear foundation,89700,0.393269
4,acne armor cover correct skin tint,39920,0.000000
5,acne armor flawless flex compact powder,43912,0.031537
6,acne armor puff perfection cushion,52712,0.101059
7,acne blur and set loose powder,109200,0.547322
8,acne cover smooth two way cake,109200,0.547322
9,airy liquid blush,54450,0.114789


results are exported to price_preprocessing.xlsx


Rating Preprocessing

In [220]:
# Get name and rating as dataframe, ensuring it's a copy to avoid SettingWithCopyWarning
df_name_rating = df[['name', 'Rating']].copy()

# Calculate min and max rating for normalization
min_rating = df_name_rating['Rating'].min()
max_rating = df_name_rating['Rating'].max()

# Apply the new normalization formula: (rating - min_rating) / (max_rating - min_rating)
# Handle division by zero if all ratings are the same
if (max_rating - min_rating) != 0:
    df_name_rating['normalized_rating'] = (df_name_rating['Rating'] - min_rating) / (max_rating - min_rating)
else:
    df_name_rating['normalized_rating'] = 0.0 # Assign 0 if all ratings are identical

# Display the DataFrame with the new column
display(df_name_rating)

# Export Excel to a single sheet
with pd.ExcelWriter('rating_preprocessing.xlsx') as writer:
    start_row = 0

    # Write rating Extraction Data
    pd.DataFrame([['Rating Extraction Data:']]).to_excel(writer, sheet_name='All Data', startrow=start_row, header=False, index=False)
    start_row += 2
    df_name_rating.to_excel(writer, sheet_name='All Data', startrow=start_row, index=False)
    start_row += len(df_name_rating) + 2

print("results are exported to rating_preprocessing.xlsx")

,name,Rating,normalized_rating
0,refill studio tropik dreamsetter airbrush sett...,4.8,0.8
1,in perfect pair cushion powder,5.0,1.0
2,hr oil control pearlfect cover skintint,4.5,0.5
3,acne th longwear foundation,5.0,1.0
4,acne armor cover correct skin tint,4.5,0.5
5,acne armor flawless flex compact powder,4.8,0.8
6,acne armor puff perfection cushion,5.0,1.0
7,acne blur and set loose powder,4.0,0.0
8,acne cover smooth two way cake,4.8,0.8
9,airy liquid blush,4.7,0.7


results are exported to rating_preprocessing.xlsx


**Prepare for Merging All Prerocessing Data**

In [221]:
# 1. Extract the last row of the df_nama_produk DataFrame, selecting columns from index 2 up to, but not including, the last column.
list_idf_produk_series = df_nama_produk.iloc[-1, 2:-1]
# 2. Convert list_idf_produk_series into a new DataFrame named list_idf_produk_df.
list_idf_produk_df = list_idf_produk_series.to_frame().T
# 3. Insert a new column named 'Product Number' at the beginning (index 0) of list_idf_produk_df and set its value to 'IDF'.
list_idf_produk_df.insert(0, 'Product Number', 'IDF')
# 4. Insert another new column named 'Product Name' at index 1 of list_idf_produk_df and set its value to an empty string.
list_idf_produk_df.insert(1, 'Product Name', '')


# --- Prepare Brand IDF for Export ---
list_idf_brand_series = df_nama_brand.iloc[-1, 2:-1]
list_idf_brand_df = pd.DataFrame([list_idf_brand_series.values], columns=list_idf_brand_series.index)
list_idf_brand_df.insert(0, 'Brand Number', 'IDF')
list_idf_brand_df.insert(1, 'Brand Name', '')
print("list_idf_brand_df created successfully:")

# --- Prepare Shades IDF for Export ---
list_idf_shade_series = df_nama_shade.iloc[-1, 2:-1]
list_idf_shade_df = pd.DataFrame([list_idf_shade_series.values], columns=list_idf_shade_series.index)
list_idf_shade_df.insert(0, 'Shade Number', 'IDF')
list_idf_shade_df.insert(1, 'Shade Name', '')
print("list_idf_shade_df created successfully:")

# --- Prepare Ingredient IDF for Export ---
list_idf_ingredient_series = df_nama_ingredient.iloc[-1, 2:-1]
list_idf_ingredient_df = pd.DataFrame([list_idf_ingredient_series.values], columns=list_idf_ingredient_series.index)
list_idf_ingredient_df.insert(0, 'Ingredient Number', 'IDF')
list_idf_ingredient_df.insert(1, 'Ingredient Name', '')

print("list_idf_ingredient_df created successfully:")
print(list_idf_ingredient_df.head())

list_idf_brand_df created successfully:
list_idf_shade_df created successfully:
list_idf_ingredient_df created successfully:
  Ingredient Number Ingredient Name  acetate  acid  acrylatedimethicone  \
0               IDF                      2.2   1.0                 11.0   

   acrylates  acrylatesstearyl  alcohol  algin     alkyl  ...      urea  \
0       11.0              11.0      5.5   11.0  3.666667  ...  3.666667   

   utamavitamin  vegan  virginiana  vulgare  water  wax   xanthan  yellow  \
0          11.0   11.0        11.0     11.0   2.75  5.5  3.666667    11.0   

       zinc  
0  1.833333  

[1 rows x 236 columns]


**Pivoting TF-IDF Dataframe**

Pivoting, now the head of column is product code (P1, P2, etc...) and Product Name

In [222]:
print("Pivoting TF-IDF and IDF DataFrames...")

# --- Pivot df_nama_produk_inverse_doc_frequency (TF-IDF for Products) ---
df_produk_tfidf_pivoted = df_nama_produk_inverse_doc_frequency.copy()
# Create a combined identifier for columns: 'Product Number - Product Name'
df_produk_tfidf_pivoted['Product Identifier'] = df_produk_tfidf_pivoted['Product Number'] + ' - ' + df_produk_tfidf_pivoted['Product Name']
# Set the combined identifier as the index and transpose, selecting only the term columns
df_produk_tfidf_pivoted = df_produk_tfidf_pivoted.set_index('Product Identifier').iloc[:, 2:].T
df_produk_tfidf_pivoted.index.name = 'Term'
print("Pivoted df_nama_produk_inverse_doc_frequency:")
display(df_produk_tfidf_pivoted.head())

# --- Pivot df_nama_brand_inverse_doc_frequency (TF-IDF for Brands) ---
df_brand_tfidf_pivoted = df_nama_brand_inverse_doc_frequency.copy()
df_brand_tfidf_pivoted['Brand Identifier'] = df_brand_tfidf_pivoted['Brand Number'] + ' - ' + df_brand_tfidf_pivoted['Brand Name']
df_brand_tfidf_pivoted = df_brand_tfidf_pivoted.set_index('Brand Identifier').iloc[:, 2:].T
df_brand_tfidf_pivoted.index.name = 'Term'

# --- Pivot df_nama_shade_inverse_doc_frequency (TF-IDF for Shades) ---
df_shade_tfidf_pivoted = df_nama_shade_inverse_doc_frequency.copy()
df_shade_tfidf_pivoted['Shade Identifier'] = df_shade_tfidf_pivoted['Shade Number'] + ' - ' + df_shade_tfidf_pivoted['Shade Name']
df_shade_tfidf_pivoted = df_shade_tfidf_pivoted.set_index('Shade Identifier').iloc[:, 2:].T
df_shade_tfidf_pivoted.index.name = 'Term'

# --- Pivot df_nama_ingredient_inverse_doc_frequency (TF-IDF for Ingredients) ---
df_ingredient_tfidf_pivoted = df_nama_ingredient_inverse_doc_frequency.copy()
df_ingredient_tfidf_pivoted['Ingredient Identifier'] = df_ingredient_tfidf_pivoted['Ingredient Number'] + ' - ' + df_ingredient_tfidf_pivoted['Ingredient Name']
df_ingredient_tfidf_pivoted = df_ingredient_tfidf_pivoted.set_index('Ingredient Identifier').iloc[:, 2:].T
df_ingredient_tfidf_pivoted.index.name = 'Term'


Pivoting TF-IDF and IDF DataFrames...
Pivoted df_nama_produk_inverse_doc_frequency:


Product Identifier,P1 - refill studio tropik dreamsetter airbrush setting powder,P2 - in perfect pair cushion powder,P3 - hr oil control pearlfect cover skintint,P4 - acne th longwear foundation,P5 - acne armor cover correct skin tint,P6 - acne armor flawless flex compact powder,P7 - acne armor puff perfection cushion,P8 - acne blur and set loose powder,P9 - acne cover smooth two way cake,P10 - airy liquid blush,P11 - airy poreless fluid foundation
Term,,,,,,,,,,,
acne,0.000000,0.0,0.0,0.458333,0.305556,0.305556,0.366667,0.305556,0.305556,0.000000,0.000
airbrush,1.571429,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000
airy,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.833333,1.375
and,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1.833333,0.000000,0.000000,0.000
armor,0.000000,0.0,0.0,0.000000,0.611111,0.611111,0.733333,0.000000,0.000000,0.000000,0.000


In [223]:
output_pivoted_excel_file = 'pivoted_preprocessing_data.xlsx'

with pd.ExcelWriter(output_pivoted_excel_file, engine='xlsxwriter') as writer:
    df_produk_tfidf_pivoted.to_excel(writer, sheet_name='Product_TFIDF')
    df_ingredient_tfidf_pivoted.to_excel(writer, sheet_name='Ingredient_TFIDF')
    df_brand_tfidf_pivoted.to_excel(writer, sheet_name='Brand_TFIDF')
    df_shade_tfidf_pivoted.to_excel(writer, sheet_name='Shade_TFIDF')

print(f"All pivoted data has been exported to '{output_pivoted_excel_file}'")

All pivoted data has been exported to 'pivoted_preprocessing_data.xlsx'


In [224]:
df_produk_tfidf_pivoted = df_produk_tfidf_pivoted.reset_index()
df_produk_tfidf_pivoted['Group'] = 'Product'
# Reorder columns: 'Group' first, 'Term' second, then other columns
other_columns = [col for col in df_produk_tfidf_pivoted.columns if col not in ['Group', 'Term']]
df_produk_tfidf_pivoted = df_produk_tfidf_pivoted[['Group', 'Term'] + other_columns]

df_ingredient_tfidf_pivoted = df_ingredient_tfidf_pivoted.reset_index()
df_ingredient_tfidf_pivoted['Group'] = 'Ingredient'
other_columns = [col for col in df_ingredient_tfidf_pivoted.columns if col not in ['Group', 'Term']]
df_ingredient_tfidf_pivoted = df_ingredient_tfidf_pivoted[['Group', 'Term'] + other_columns]

df_brand_tfidf_pivoted = df_brand_tfidf_pivoted.reset_index()
df_brand_tfidf_pivoted['Group'] = 'Brand'
other_columns = [col for col in df_brand_tfidf_pivoted.columns if col not in ['Group', 'Term']]
df_brand_tfidf_pivoted = df_brand_tfidf_pivoted[['Group', 'Term'] + other_columns]

df_shade_tfidf_pivoted = df_shade_tfidf_pivoted.reset_index()
df_shade_tfidf_pivoted['Group'] = 'Shade'
other_columns = [col for col in df_shade_tfidf_pivoted.columns if col not in ['Group', 'Term']]
df_shade_tfidf_pivoted = df_shade_tfidf_pivoted[['Group', 'Term'] + other_columns]

print("All pivoted DataFrames have been processed with reset index and 'Group' column.")

All pivoted DataFrames have been processed with reset index and 'Group' column.


In [225]:
output_excel_file = 'all_pivoted_data.xlsx'
with pd.ExcelWriter(output_excel_file, engine='xlsxwriter') as writer:
    start_row = 0

    # Products
    df_produk_tfidf_pivoted.to_excel(writer, sheet_name='All Pivoted Data Merged', startrow=start_row, index=False)
    start_row += len(df_produk_tfidf_pivoted)

    # Brands
    df_brand_tfidf_pivoted.to_excel(writer, sheet_name='All Pivoted Data Merged', startrow=start_row, index=False, header=False)
    start_row += len(df_brand_tfidf_pivoted)

    # Shades
    df_shade_tfidf_pivoted.to_excel(writer, sheet_name='All Pivoted Data Merged', startrow=start_row, index=False, header=False)
    start_row += len(df_shade_tfidf_pivoted)

    # Ingredients
    df_ingredient_tfidf_pivoted.to_excel(writer, sheet_name='All Pivoted Data Merged', startrow=start_row, index=False, header=False)

print(f"All preprocessed data has been consolidated into '{output_excel_file}'")

All preprocessed data has been consolidated into 'all_pivoted_data.xlsx'


**Merged All Preprocessing to Single Excel File**

In [226]:
output_excel_file = 'all_preprocessing_data.xlsx'
with pd.ExcelWriter(output_excel_file, engine='xlsxwriter') as writer:
    start_row = 0
    # Product Data
    pd.DataFrame([['Product Extraction Data:']]).to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk.to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_produk) + 2

    pd.DataFrame([['Product IDF Data:']]).to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    list_idf_produk_df.to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, index=False)
    start_row += len(list_idf_produk_df) + 2

    pd.DataFrame([['Product Term Frequency Data:']]).to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk_term_frequency.to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_produk_term_frequency) + 2

    pd.DataFrame([['Product TF-IDF Data:']]).to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_produk_inverse_doc_frequency.to_excel(writer, sheet_name='Product Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Ingredient Data
    pd.DataFrame([['Ingredient Extraction Data:']]).to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient.to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient) + 2

    pd.DataFrame([['Ingredient Term Frequency Data:']]).to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient_term_frequency.to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient_term_frequency) + 2

    pd.DataFrame([['Ingredient TF-IDF Data:']]).to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_ingredient_inverse_doc_frequency.to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_ingredient_inverse_doc_frequency) + 2

    pd.DataFrame([['Ingredient IDF Data:']]).to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    list_idf_ingredient_df.to_excel(writer, sheet_name='Ingredient Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Brand Data
    pd.DataFrame([['Brand Extraction Data:']]).to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand.to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_brand) + 2

    pd.DataFrame([['Brand Term Frequency Data:']]).to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand_term_frequency.to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_brand_term_frequency) + 2

    pd.DataFrame([['Brand TF-IDF Data:']]).to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_brand_inverse_doc_frequency.to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_brand_inverse_doc_frequency) + 2

    pd.DataFrame([['Brand IDF Data:']]).to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    list_idf_brand_df.to_excel(writer, sheet_name='Brand Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Shades Data
    pd.DataFrame([['Shade Extraction Data:']]).to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade.to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_shade) + 2

    pd.DataFrame([['Shade Term Frequency Data:']]).to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade_term_frequency.to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_shade_term_frequency) + 2

    pd.DataFrame([['Shade TF-IDF Data:']]).to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_shade_inverse_doc_frequency.to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, index=False)
    start_row += len(df_nama_shade_inverse_doc_frequency) + 2

    pd.DataFrame([['Shade IDF Data:']]).to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    list_idf_shade_df.to_excel(writer, sheet_name='Shade Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Category Data
    pd.DataFrame([['Category Extraction Data:']]).to_excel(writer, sheet_name='Category Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_nama_category.to_excel(writer, sheet_name='Category Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Name and Price Data
    pd.DataFrame([['Price Extraction Data:']]).to_excel(writer, sheet_name='Price Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_name_price.to_excel(writer, sheet_name='Price Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Name and Rating Data
    pd.DataFrame([['Rating Extraction Data:']]).to_excel(writer, sheet_name='Rating Preprocessing', startrow=start_row, header=False, index=False)
    start_row += 2
    df_name_rating.to_excel(writer, sheet_name='Rating Preprocessing', startrow=start_row, index=False)

    start_row = 0
    # Pivot and Merged All Preprocessing Data
    pd.DataFrame([['Pivot and Merged Preprocessing Data:']]).to_excel(writer, sheet_name='All Merged Preprocessing Data', startrow=start_row, header=False, index=False)
    start_row += 2

    # Products
    df_produk_tfidf_pivoted.to_excel(writer, sheet_name='All Merged Preprocessing Data', startrow=start_row, index=False)
    start_row += len(df_produk_tfidf_pivoted)

    # Brands
    df_brand_tfidf_pivoted.to_excel(writer, sheet_name='All Merged Preprocessing Data', startrow=start_row, index=False, header=False)
    start_row += len(df_brand_tfidf_pivoted)

    # Shades
    df_shade_tfidf_pivoted.to_excel(writer, sheet_name='All Merged Preprocessing Data', startrow=start_row, index=False, header=False)
    start_row += len(df_shade_tfidf_pivoted)

    # Ingredients
    df_ingredient_tfidf_pivoted.to_excel(writer, sheet_name='All Merged Preprocessing Data', startrow=start_row, index=False, header=False)

print(f"All preprocessed data has been consolidated into '{output_excel_file}'")

All preprocessed data has been consolidated into 'all_preprocessing_data.xlsx'
